In [16]:
using Combinatorics, ClusteredLowRankSolver, Infinity, LinearAlgebra, Nemo
include("fun.jl")

# setprecision(10^3)

LDL (generic function with 1 method)

In [17]:
# [7,1,4], [19,1,8], [13,1,6]

n,K,d = [7,1,4]

R, x = polynomial_ring(QQ, :x)
N, z = number_field(x^2 - 3, :z)

name_file=string("cerf/selfdual/$n","_$K","_$d.jl")

include(name_file) # includes Y-> set of positive matrices

20-element Vector{Matrix{Any}}:
 [3466*z + 121251 0 … -366*z - 25199 -25543*z - 1129; 0 -1730*z + 395424 … 0 0; … ; -366*z - 25199 0 … -z + 7057 12954; -25543*z - 1129 0 … 12954 2*z + 35184]
 [54*z + 224248 0 … 0 0; 0 -109*z + 428267 … 0 0; … ; 0 0 … 3*z + 16010 9665; 0 0 … 9665 -5*z + 32045]
 [130*z + 130270 0 0 0; 0 2*z + 379267 0 0; 0 0 -16*z + 45196 21706; 0 0 21706 18*z + 313491]
 [12*z + 472502 0; 0 -168235//1016*z + 41469875//1016]
 [-17*z + 250691 0 … 0 0; 0 -17*z + 250691 … 0 0; … ; 0 0 … -7*z + 19436 35372; 0 0 … 35372 57*z + 166018]
 [-z + 58073 0 … 0 0; 0 -20*z + 116137 … 0 0; … ; 0 0 … 52*z + 339034 192099; 0 0 … 192099 -15*z + 343474]
 [12*z + 138831 0 0; 0 -647961//2032*z + 293460849//2032 270144; 0 270144 -65442//635*z + 1409260077//2540]
 [-296759//508*z + 60695135//508;;]
 [-61*z + 120798 0 … 0 0; 0 -54*z + 91793 … 0 0; … ; 0 0 … -38*z + 166901 119685; 0 0 … 119685 210*z + 376786]
 [-136*z + 290328 0 0 0; 0 -18*z + 262443 96343 34801; 0 96343 44*z + 65792 51581; 0 348

In [18]:
f4, size = f4_fun(n)
f2, size2 = f2_fun(n)

y_field=yijtp_fun_field(Y, n, f2, z)

w_field=(-2*y_field[f4[n, 0, 0, 0]+1]-y_field[f4[n, n, n, n]+1])/gamma_field(n,0,0,0,n)

# Obj. funct

obj_fun = (2^n-1)*w_field-y_field[f4[0, 0, 0, 0]+1]

println("Objective function = $obj_fun")
println("\n")

# Constraint 1

ref=0

for i in d:n-1
    max=(gamma_field(i,i,i,i,n)*w_field+(2*y_field[f4[i, 0, 0, 0]+1]+y_field[f4[i, i, i, i]+1]))
    max=BigFloat(coeff(max,0))+BigFloat(coeff(max,1))*sqrt(big(3))
    max=abs(max)
    
    if max > ref
        ref=max
    end
end

println("Min value of the constraint 1 = $ref")
println("\n")

# Constraint 2

ref=0 

for (i, j, t, p) in keys(f4)
    if ((t - p) % 2 == 0 && i !=0 && j != 0 && i+j-t-p >= d)

        max=y_field[f4[i,j,t,p]+1]

        max=BigFloat(coeff(max,0))+BigFloat(coeff(max,1))*sqrt(big(3))
        
        max=abs(max)
        
        if max > ref
            ref=max
        end
    end
end

println("Min value of the constraint 2 = $ref")

Objective function = 1


Min value of the constraint 1 = 0


Min value of the constraint 2 = 0


In [19]:
# Maximum and minimum eigenvalue of Y

Ypsd=[]

for k in 1:size2
    dim=length(Y[k][1,:])
    
    M=Matrix(undef,dim,dim)
    
    for i in 1:dim
        for j in 1:dim
            
            if typeof(Y[k][i,j])==AbsSimpleNumFieldElem
                
                M[i,j]=BigFloat(coeff(Y[k][i,j],0))+BigFloat(coeff(Y[k][i,j],1))*sqrt(big(3))
            else
                M[i,j]=Y[k][i,j]
               
        
            end
        end
    end
    
    push!(Ypsd,M)
    
end

ref_min=Inf
ref_max=0
for i in 1:size2
    mat=Matrix{BigFloat}(Ypsd[i])
    
    eigen=eigvals(mat)

    eigen_abs=abs.(eigen)

    min=minimum(eigen_abs)
    max=maximum(eigen_abs)

    if ref_min > min
        ref_min=min
    end
    if max > ref_max
        ref_max=max
    end
end

println("Minimum eigenvalue = $ref_min\n")
println("Maximum eigenvalue = $ref_max")

Minimum eigenvalue = 9.490496047725463885619439294286802462970458181917783778829307767368719282262536

Maximum eigenvalue = 839916.9400038313535373516954941776533388871869876916711918363881924271260423511


In [20]:
# Check positivity with LDL method

ref_min=Inf

upper=rationalize(1.73206)
lower=rationalize(1.73205) 

for k in 1:size2
    A=Y[k]
    D=length(A[1,:])
    d=LDL(A,D)
    d_factor=Matrix(undef,D,1)

    for j in 1:D

        if coeff(d[j],1) < 0
            factor=coeff(d[j],0)+upper*coeff(d[j],1) # bigger value of sqrt(3)
        else
            factor=coeff(d[j],0)+lower*coeff(d[j],1) # smaller value of sqrt(3)
        end
        
        d_factor[j]=factor
    end
    
    ref=minimum(d_factor)

    if ref_min > ref
        ref_min=ref
    end
end

ref_min_float=BigFloat(ref_min)

println("Minimum value (Fraction) = $ref_min")
println("Minimum value (Float) = $ref_min_float")

Minimum value (Fraction) = 25640604327334473//97771770220000
Minimum value (Float) = 262.2495662054555055595269347880689318258719771392925077387434869745779672966224
